# jA_02 (SIM) - FL near-domain, simulasi penuh di SageMaker (JURNAL, Konfig A)

Menjalankan **federated learning near-domain** (Client-CIC2017 + Client-CIC2018,
~76 fitur) sebagai **simulasi FedAvg di satu kernel SageMaker**. Tidak memakai EC2
atau CloudFormation, jadi tidak butuh izin `cloudformation:CreateStack`.

**Mengapa simulasi sah di sini:** near-domain mengukur *hasil* FL (MCC federated vs
centralized vs local-only), bukan bukti deployment cloud. Bukti deployment EC2 nyata
sudah ada di jalur cross-domain (ISRITI R3/R5/R6). Di naskah jurnal, near-domain
ditulis sebagai *simulasi FL terkontrol* (konsisten dgn R1/R2 ISRITI yang juga simulasi);
matematika FedAvg, z-score per-klien, dan arsitektur MLP IDENTIK dgn node EC2
(`fed_node_journal.py`).

**Protokol evaluasi bebas-kebocoran (kontribusi jurnal):** tiap test set dinormalisasi
dgn statistik TRAIN klien pemiliknya (match training). Centralized dilaporkan dua cara
(pooled-norm & local-norm) agar efek normalisasi terpisah dari efek federasi.

**Prasyarat:** `jA_01_partition_neardomain.ipynb` sudah jalan -> partisi di S3
`federated_journal/neardomain/data/{cic2017,cic2018}/`.

**Output:** `federated_journal/neardomain/run_J_NEAR_s<seed>/eval_results.json` (di-upload
ke S3) + agregasi multi-seed `journal_out/neardomain_fed_{seeds,agg}.csv` lokal.

> Jalankan di SageMaker. Angka dari eksekusi nyata; JANGAN mengarang.

## Sel 1 - Setup + muat data near-domain dari S3

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','boto3','scipy'):
    if importlib.util.find_spec(m) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, numpy as np, boto3

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='federated_journal/neardomain'
CLIENTS=['cic2017','cic2018']
OUTDIR='journal_out'; os.makedirs(OUTDIR, exist_ok=True)
s3=boto3.client('s3',region_name=REGION)

def s3_get_npz(key):
    obj=s3.get_object(Bucket=BUCKET,Key=key)
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
def s3_put_bytes(key,data):
    s3.put_object(Bucket=BUCKET,Key=key,Body=data)

def load_client(cid, split):
    d=s3_get_npz(f'{BASE}/data/{cid}/{cid}_{split}.npz')
    return np.asarray(d['X'],float), np.asarray(d['y'],int)

TR={}; TE={}
for c in CLIENTS:
    TR[c]=load_client(c,'train'); TE[c]=load_client(c,'test')
    print(c,'train',TR[c][0].shape,'pos=%.3f'%TR[c][1].mean(),
          '| test',TE[c][0].shape,'pos=%.3f'%TE[c][1].mean())
IN_DIM=TR[CLIENTS[0]][0].shape[1]
assert all(TR[c][0].shape[1]==IN_DIM for c in CLIENTS), 'dim fitur antar-klien tak sama!'
print('input dim =',IN_DIM)

## Sel 2 - MLP numpy + FedAvg + metrik (identik node EC2)

In [ ]:
HID=[128,64]
def layers_for(d): return [d]+HID+[1]
def init_weights(d, seed=42):
    rng=np.random.RandomState(seed); W=[]; L=layers_for(d)
    for a,b in zip(L[:-1],L[1:]):
        W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; out=_sig(a3).ravel()
    return out,(X,a1,z1,a2,z2,a3,out)
def train_local(W,X,y,epochs=1,lr=3e-3,batch=256,mu=0.0,W_global=None,seed=42):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2,a3,o)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0)
            gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and W_global is not None: gv=gv+mu*(W[i]-W_global[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(wl,sizes):
    tot=float(sum(sizes)); out=[]
    for i in range(len(wl[0])):
        out.append(sum((sizes[k]/tot)*wl[k][i] for k in range(len(wl))))
    return out
def zscore_fit(X):
    mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zscore_apply(X,mu,sd): return (X-mu)/sd
def mcc(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); tn=int(((p==0)&(y==0)).sum())
    fp=int(((p==1)&(y==0)).sum()); fn=int(((p==0)&(y==1)).sum())
    den=np.sqrt(float(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den>0 else 0.0
def recall(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); fn=int(((p==0)&(y==1)).sum())
    return tp/(tp+fn) if (tp+fn)>0 else 0.0
def metrics(y,prob):
    p=(prob>=0.5).astype(int)
    return {'mcc':round(mcc(y,p),4),'recall':round(recall(y,p),4),
            'n':int(len(y)),'pos_rate':round(float(np.mean(y)),4)}
print('model + util siap')

## Sel 3 - Satu run FL (1 seed) + evaluasi bebas-kebocoran

FedAvg 100 ronde, 5 epoch lokal/ronde, z-score per-klien lokal. Lalu evaluasi:
federated (local-norm utama + pooled diagnostik), centralized (pooled & local-norm),
local-only tiap klien.

In [ ]:
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3

def run_one_seed(seed):
    # z-score per-klien dari TRAIN masing-masing
    zstat={c:zscore_fit(TR[c][0]) for c in CLIENTS}
    Xz={c:zscore_apply(TR[c][0],*zstat[c]) for c in CLIENTS}
    # --- FedAvg ---
    W=init_weights(IN_DIM, seed)
    for t in range(ROUNDS):
        cw=[]; cs=[]
        for c in CLIENTS:
            Wl=train_local(W,Xz[c],TR[c][1],epochs=LOCAL_EPOCHS,lr=LR,W_global=W,seed=seed)
            cw.append(Wl); cs.append(len(TR[c][1]))
        W=fedavg(cw,cs)
    W_fed=W

    def eval_model(Wm, norm='local'):
        if norm=='pooled':
            Xall=np.vstack([TR[c][0] for c in CLIENTS]); mup,sdp=zscore_fit(Xall)
        per={}; pa=[]; ya=[]
        for c in CLIENTS:
            Xte,yte=TE[c]
            mz,sz=(zstat[c] if norm=='local' else (mup,sdp))
            prob,_=forward(Wm, zscore_apply(Xte,mz,sz))
            per[c]=metrics(yte,prob); pa.append(prob); ya.append(yte)
        return {'per_client':per,'union':metrics(np.concatenate(ya),np.concatenate(pa))}

    res={'run':f'J_NEAR_s{seed}','clients':CLIENTS,'in_dim':int(IN_DIM),'seed':int(seed),
         'rounds':ROUNDS,'local_epochs':LOCAL_EPOCHS,'mode':'simulation(SageMaker)'}
    res['federated_local']=eval_model(W_fed,'local')
    res['federated_pooled_diagnostic']=eval_model(W_fed,'pooled')
    # centralized (pooled train)
    Xcat=np.vstack([TR[c][0] for c in CLIENTS]); ycat=np.concatenate([TR[c][1] for c in CLIENTS])
    mup,sdp=zscore_fit(Xcat)
    Wc=init_weights(IN_DIM,seed)
    Wc=train_local(Wc,zscore_apply(Xcat,mup,sdp),ycat,epochs=ROUNDS,lr=LR,seed=seed)
    res['centralized_pooled']=eval_model(Wc,'pooled')
    res['centralized_localnorm']=eval_model(Wc,'local')
    # local-only tiap klien
    for c in CLIENTS:
        mz,sz=zstat[c]
        Wl=init_weights(IN_DIM,seed)
        Wl=train_local(Wl,zscore_apply(TR[c][0],mz,sz),TR[c][1],epochs=ROUNDS,lr=LR,seed=seed)
        res[f'local_only[{c}]']=eval_model(Wl,'local')
    return res

print('fungsi run_one_seed siap (ROUNDS=%d)'%ROUNDS)

## Sel 4 - Jalankan 5 seed + upload eval_results.json ke S3

In [ ]:
import time
SEEDS=[13,42,101,202,303]
all_res={}
for sd in SEEDS:
    t0=time.time(); r=run_one_seed(sd); dt=time.time()-t0
    all_res[sd]=r
    fl=r['federated_local']['union']['mcc']; cp=r['centralized_pooled']['union']['mcc']
    cl=r['centralized_localnorm']['union']['mcc']; fpd=r['federated_pooled_diagnostic']['union']['mcc']
    print(f'seed {sd}: fed(local)={fl}  cent(local)={cl}  cent(pooled)={cp}  fed(pooled-diag)={fpd}  [{dt:.0f}s]')
    # upload per-seed json ke S3 (format setara jalur EC2)
    s3_put_bytes(f'{BASE}/run_J_NEAR_s{sd}/eval_results.json',
                 json.dumps(r,indent=2).encode('utf-8'))
print('\nselesai + ter-upload ke s3://%s/%s/run_J_NEAR_s*/eval_results.json'%(BUCKET,BASE))

## Sel 5 - Agregasi multi-seed (mean +/- CI95) + simpan CSV + upload

CI95 via t-distribusi (n=5). CSV ini yang Kiro tarik untuk mengisi tabel jurnal.

In [ ]:
import csv
from scipy import stats

# kumpulkan metrik union + per-client untuk tiap 'model'
MODELS=['federated_local','centralized_localnorm','centralized_pooled',
        'federated_pooled_diagnostic','local_only[cic2017]','local_only[cic2018]']

def collect(metric_path):
    """metric_path: (model, scope, metric) -> list across seeds. scope='union' atau client id."""
    model,scope,metric=metric_path
    vals=[]
    for sd in SEEDS:
        node=all_res[sd][model]
        m=(node['union'] if scope=='union' else node['per_client'][scope])
        vals.append(m[metric])
    return vals

def mean_ci95(vals):
    a=np.asarray(vals,float); n=len(a); mu=a.mean()
    if n<2: return mu,0.0
    se=a.std(ddof=1)/np.sqrt(n); h=se*stats.t.ppf(0.975,n-1)
    return mu,h

# tulis seeds (raw) + agg
rows_seeds=[]; rows_agg=[]
scopes=['union']+CLIENTS
for model in MODELS:
    for scope in scopes:
        for metric in ['mcc','recall']:
            vals=collect((model,scope,metric))
            for sd,v in zip(SEEDS,vals):
                rows_seeds.append({'model':model,'scope':scope,'metric':metric,'seed':sd,'value':v})
            mu,h=mean_ci95(vals)
            rows_agg.append({'model':model,'scope':scope,'metric':metric,
                             'mean':round(mu,4),'ci95':round(h,4),
                             'values':';'.join(str(v) for v in vals)})

def write_csv(path,rows,fields):
    with open(path,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
    print('tulis',path,len(rows),'baris')
write_csv(os.path.join(OUTDIR,'neardomain_fed_seeds.csv'),rows_seeds,
          ['model','scope','metric','seed','value'])
write_csv(os.path.join(OUTDIR,'neardomain_fed_agg.csv'),rows_agg,
          ['model','scope','metric','mean','ci95','values'])

# upload agregasi ke S3 (meta)
for f in ('neardomain_fed_seeds.csv','neardomain_fed_agg.csv'):
    s3.upload_file(os.path.join(OUTDIR,f),BUCKET,f'{BASE}/results/{f}')
print('upload agregasi -> s3://%s/%s/results/'%(BUCKET,BASE))

# tampilkan ringkas union MCC
print('\n=== RINGKAS (union MCC, mean +/- CI95) ===')
for r in rows_agg:
    if r['scope']=='union' and r['metric']=='mcc':
        print(f"  {r['model']:32s} {r['mean']:+.3f} +/- {r['ci95']:.3f}")

## Catatan untuk naskah jurnal

- Near-domain = **simulasi FL terkontrol** di SageMaker (matematika FedAvg, z-score
  per-klien, MLP identik dgn node EC2 `fed_node_journal.py`). Bukti deployment cloud
  nyata ada di jalur cross-domain (ISRITI R3/R5/R6).
- Protokol evaluasi **bebas-kebocoran**: test set dinormalisasi dgn statistik train
  klien pemiliknya; centralized dilaporkan pooled-norm & local-norm untuk memisah
  efek normalisasi dari efek federasi; federated-pooled dilaporkan sbg DIAGNOSTIK.
- 5 seed {13,42,101,202,303}; CI95 via t. Partisi tetap (dari jA_01) -> variasi =
  keacakan training saja.
- Angka untuk tabel jurnal diambil dari `neardomain_fed_agg.csv` (hasil nyata).